# 02 — Data Cleaning
**AutoAnalyst | Finance Module**

Cleans any finance dataset based on its detected type.
- `timeseries`    → date parsing, missing price fill, type fixes
- `transactional` → date parsing, category standardization, outlier flagging
- `fundamental`   → numeric coercion, missing value strategy, column cleanup

Input  : `df` + `ingestion_report` from `01_data_ingestion.ipynb`
Output : `df_clean` + `cleaning_report`

In [26]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded.')

✅ Libraries loaded.


In [27]:
# ── Cell 2: Config — change DATASET_FILENAME to switch datasets ───────────────
#
# Available:
#   RELIANCE.NS.csv
#   Credit card transactions - India - Simple.csv
#   Annual_P_L_1_final.csv  /  Annual_P_L_2_final.csv
#   Quarter_P_L_1_final.csv /  Quarter_P_L_2_final.csv
#   Balance_Sheet_final.csv
#   cash_flow_statments_final.csv
#   ratios_1_final.csv      /  ratios_2_final.csv
#   other_metrics_final.csv
#   price_final.csv         /  t1_prices.csv

DATASET_FILENAME = 'Annual_P_L_1_final.csv'
BASE_PATH        = r'R:\AutoAnalyst\finance_module\datasets'
DATASET_PATH     = os.path.join(BASE_PATH, DATASET_FILENAME)

print(f'Dataset : {DATASET_FILENAME}')
print(f'Exists  : {os.path.exists(DATASET_PATH)}')

Dataset : Annual_P_L_1_final.csv
Exists  : True


In [28]:
# ── Cell 3: Smart Loader (copied from 01_data_ingestion) ──────────────────────
# Each notebook is self-contained — we redefine the loader here
# so this notebook runs independently without needing notebook 01 open.

def smart_load(filepath):
    raw = pd.read_csv(filepath, nrows=3, header=0)
    first_val = str(raw.iloc[0, 0]).strip().lower()
    metadata_keywords = ['ticker', 'date', 'symbol', 'name', 'description']

    if first_val in metadata_keywords:
        df = pd.read_csv(filepath, skiprows=[1, 2], header=0)
        df.rename(columns={df.columns[0]: 'Date'}, inplace=True)
        print(f'[smart_load] Metadata rows skipped.')
    else:
        df = pd.read_csv(filepath, header=0)
        print(f'[smart_load] Clean header. Loaded normally.')

    return df


def detect_finance_type(df):
    cols_str = ' '.join([c.lower().strip() for c in df.columns])
    timeseries_kw    = ['close', 'open', 'high', 'low', 'volume', 'price', 'adj close']
    transactional_kw = ['transaction', 'amount', 'debit', 'credit', 'merchant',
                        'category', 'expense', 'card', 'payment', 'exp type']
    fundamental_kw   = ['revenue', 'profit', 'ebitda', 'eps', 'assets', 'liabilities',
                        'equity', 'net profit', 'bse', 'nse', 'market cap', 'ratio',
                        'roce', 'roe', 'sales', 'turnover', 'earnings']

    ts = sum(1 for kw in timeseries_kw    if kw in cols_str)
    tx = sum(1 for kw in transactional_kw if kw in cols_str)
    fn = sum(1 for kw in fundamental_kw   if kw in cols_str)
    scores = {'timeseries': ts, 'transactional': tx, 'fundamental': fn}
    best   = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'unknown'


df_raw       = smart_load(DATASET_PATH)
dataset_type = detect_finance_type(df_raw)
print(f'Type    : {dataset_type.upper()}')
print(f'Shape   : {df_raw.shape}')

[smart_load] Clean header. Loaded normally.
Type    : FUNDAMENTAL
Shape   : (4668, 58)


In [29]:
# ── Cell 4: Pre-cleaning Snapshot ────────────────────────────────────────────
# Always compare before vs after cleaning.
# This captures the state before we touch anything.

def snapshot(df, label):
    print(f'\n── {label} ──────────────────────────────')
    print(f'  Shape      : {df.shape}')
    print(f'  Duplicates : {df.duplicated().sum()}')
    total_missing = df.isnull().sum().sum()
    print(f'  Missing    : {total_missing} cells ({total_missing / df.size * 100:.2f}%)')
    print(f'  Dtypes     :')
    for col, dtype in df.dtypes.items():
        print(f'    • {col}: {dtype}')


snapshot(df_raw, 'BEFORE CLEANING')


── BEFORE CLEANING ──────────────────────────────
  Shape      : (4668, 58)
  Duplicates : 0
  Missing    : 4217 cells (1.56%)
  Dtypes     :
    • Name: str
    • BSE Code: float64
    • NSE Code: str
    • Industry: str
    • Current Price: float64
    • Sales: float64
    • OPM: float64
    • Profit after tax: float64
    • Return on capital employed: float64
    • EPS: float64
    • Change in promoter holding: float64
    • Sales last year: float64
    • Operating profit last year: float64
    • Other income last year: float64
    • EBIDT last year: float64
    • Depreciation last year: float64
    • EBIT last year: float64
    • Interest last year: float64
    • Profit before tax last year: float64
    • Tax last year: float64
    • Profit after tax last year: float64
    • Extraordinary items last year: float64
    • Net Profit last year: float64
    • Dividend last year: float64
    • Material cost last year: float64
    • Employee cost last year: float64
    • OPM last year: f

In [30]:
# ── Cell 5: Universal Cleaning (runs on ALL dataset types) ───────────────────
# These steps apply regardless of dataset type:
#   1. Remove exact duplicate rows
#   2. Strip whitespace from string columns
#   3. Standardize column names (lowercase, spaces to underscores)

def universal_clean(df):
    df = df.copy()
    before_rows = len(df)

    # 1. Drop exact duplicates
    df.drop_duplicates(inplace=True)
    dropped = before_rows - len(df)
    print(f'  Duplicates removed : {dropped}')

    # 2. Strip whitespace from all string columns
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        df[col] = df[col].str.strip()
    print(f'  Whitespace stripped from {len(str_cols)} string columns')

    # 3. Standardize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace(r'[^\w]', '_', regex=True)
    )
    print(f'  Column names standardized')

    return df


print('Running universal cleaning...')
df_clean = universal_clean(df_raw)
print(f'\nColumns after standardization:')
print(df_clean.columns.tolist())

Running universal cleaning...
  Duplicates removed : 0
  Whitespace stripped from 4 string columns
  Column names standardized

Columns after standardization:
['name', 'bse_code', 'nse_code', 'industry', 'current_price', 'sales', 'opm', 'profit_after_tax', 'return_on_capital_employed', 'eps', 'change_in_promoter_holding', 'sales_last_year', 'operating_profit_last_year', 'other_income_last_year', 'ebidt_last_year', 'depreciation_last_year', 'ebit_last_year', 'interest_last_year', 'profit_before_tax_last_year', 'tax_last_year', 'profit_after_tax_last_year', 'extraordinary_items_last_year', 'net_profit_last_year', 'dividend_last_year', 'material_cost_last_year', 'employee_cost_last_year', 'opm_last_year', 'npm_last_year', 'operating_profit', 'interest', 'depreciation', 'eps_last_year', 'ebit', 'net_profit', 'current_tax', 'tax', 'other_income', 'last_annual_result_date', 'sales_preceding_year', 'operating_profit_preceding_year', 'other_income_preceding_year', 'ebidt_preceding_year', 'depr

In [31]:
# ── Cell 6: Type-Specific Cleaning ───────────────────────────────────────────
# Each dataset type has different cleaning needs.
# The right function is called automatically based on detected type.

# ── TIMESERIES cleaner ───────────────────────────────────────────────────────
def clean_timeseries(df):
    """
    For OHLCV stock data (RELIANCE.NS, price timeseries)
    - Parse dates
    - Sort by date ascending
    - Forward-fill missing price values (standard finance practice)
    - Ensure numeric types on price/volume columns
    """
    df = df.copy()

    # Find the date column (named 'date' after standardization)
    date_col = 'date' if 'date' in df.columns else df.columns[0]

    # Parse date
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    bad_dates = df[date_col].isnull().sum()
    if bad_dates > 0:
        print(f'  ⚠️  {bad_dates} unparseable date rows dropped')
        df.dropna(subset=[date_col], inplace=True)

    # Sort chronologically
    df.sort_values(date_col, inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'  Date range : {df[date_col].min().date()} → {df[date_col].max().date()}')

    # Numeric columns — coerce anything that slipped through
    num_cols = [c for c in df.columns if c != date_col]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Missing price values — forward fill (carry last known price forward)
    # This is standard practice in finance — missing day = previous day's price
    missing_before = df[num_cols].isnull().sum().sum()
    df[num_cols] = df[num_cols].ffill()
    missing_after  = df[num_cols].isnull().sum().sum()
    print(f'  Missing prices filled : {missing_before - missing_after} (forward-fill)')

    # Flag outliers in Close price using IQR
    if 'close' in df.columns:
        Q1  = df['close'].quantile(0.25)
        Q3  = df['close'].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR
        outliers = df[(df['close'] < lower) | (df['close'] > upper)]
        df['is_price_outlier'] = ((df['close'] < lower) | (df['close'] > upper))
        print(f'  Price outliers flagged : {len(outliers)} rows (IQR method, 3x)')

    print(f'  ✅ Timeseries cleaning done.')
    return df


# ── TRANSACTIONAL cleaner ────────────────────────────────────────────────────
def clean_transactional(df):
    """
    For credit card / transaction data
    - Parse dates
    - Clean city names (remove ', India' suffix)
    - Standardize categories to title case
    - Flag amount outliers
    - Add derived time columns (month, year, day_of_week)
    """
    df = df.copy()

    # Find date column
    date_col = next((c for c in df.columns if 'date' in c.lower()), None)
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
        df.sort_values(date_col, inplace=True)
        df.reset_index(drop=True, inplace=True)
        # Derived time features — useful for analysis
        df['month']        = df[date_col].dt.month
        df['year']         = df[date_col].dt.year
        df['day_of_week']  = df[date_col].dt.day_name()
        print(f'  Date parsed. Range: {df[date_col].min().date()} → {df[date_col].max().date()}')
        print(f'  Derived columns added: month, year, day_of_week')

    # Clean city names — remove ', India' suffix if present
    city_col = next((c for c in df.columns if 'city' in c.lower()), None)
    if city_col:
        df[city_col] = df[city_col].str.replace(', India', '', regex=False).str.strip()
        print(f'  City names cleaned (removed ", India" suffix)')
        print(f'  Unique cities: {df[city_col].nunique()}')

    # Standardize categorical columns to Title Case
    cat_cols = ['exp_type', 'card_type', 'gender']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].str.title()
    print(f'  Categorical columns standardized to Title Case')

    # Drop the unnamed index column if it exists
    if 'index' in df.columns:
        df.drop(columns=['index'], inplace=True)
        print(f'  Dropped redundant index column')

    # Flag amount outliers using IQR
    amount_col = next((c for c in df.columns if 'amount' in c.lower()), None)
    if amount_col:
        Q1  = df[amount_col].quantile(0.25)
        Q3  = df[amount_col].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR
        df['is_amount_outlier'] = ((df[amount_col] < lower) | (df[amount_col] > upper))
        n_out = df['is_amount_outlier'].sum()
        print(f'  Amount outliers flagged: {n_out} transactions')

    print(f'  ✅ Transactional cleaning done.')
    return df


# ── FUNDAMENTAL cleaner ──────────────────────────────────────────────────────
def clean_fundamental(df):
    """
    For company financials (P&L, Balance Sheet, Ratios, etc.)
    - Drop columns that are >60% empty (too sparse to be useful)
    - Coerce all numeric-looking columns to float
    - Standardize company name column
    - Flag rows where core financial metrics are all missing
    - Drop join_key if present (internal merge artifact)
    """
    df = df.copy()

    # Drop join_key — it's a merge artifact, not real data
    if 'join_key' in df.columns:
        df.drop(columns=['join_key'], inplace=True)
        print(f'  Dropped join_key column')

    # Drop columns where >60% values are missing
    threshold    = 0.6
    missing_pct  = df.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
    if cols_to_drop:
        df.drop(columns=cols_to_drop, inplace=True)
        print(f'  Dropped {len(cols_to_drop)} columns with >60% missing values')
    else:
        print(f'  No columns exceeded 60% missing threshold')

    # Coerce numeric columns — some may be stored as strings
    non_numeric = ['name', 'nse_code', 'industry', 'nse code', 'bse code']
    num_candidates = [c for c in df.columns
                      if c not in non_numeric
                      and df[c].dtype == object]
    coerced = 0
    for col in num_candidates:
        converted = pd.to_numeric(df[col], errors='coerce')
        # Only replace if conversion worked for >50% of non-null values
        if converted.notna().sum() > df[col].notna().sum() * 0.5:
            df[col] = converted
            coerced += 1
    print(f'  Numeric coercion applied to {coerced} columns')

    # Flag companies where ALL key financial metrics are missing
    key_cols = [c for c in ['sales', 'net_profit', 'profit_after_tax',
                             'market_capitalization', 'eps']
                if c in df.columns]
    if key_cols:
        df['all_metrics_missing'] = df[key_cols].isnull().all(axis=1)
        n_empty = df['all_metrics_missing'].sum()
        print(f'  Companies with all key metrics missing: {n_empty}')

    print(f'  ✅ Fundamental cleaning done.')
    return df


# ── Run the right cleaner automatically ──────────────────────────────────────
print(f'Running {dataset_type.upper()} cleaner...\n')

if dataset_type == 'timeseries':
    df_clean = clean_timeseries(df_clean)
elif dataset_type == 'transactional':
    df_clean = clean_transactional(df_clean)
elif dataset_type == 'fundamental':
    df_clean = clean_fundamental(df_clean)
else:
    print('⚠️  Unknown dataset type — only universal cleaning applied.')

Running FUNDAMENTAL cleaner...

  Dropped join_key column
  No columns exceeded 60% missing threshold
  Numeric coercion applied to 0 columns
  Companies with all key metrics missing: 0
  ✅ Fundamental cleaning done.


In [32]:
# ── Cell 7: Post-cleaning Snapshot ───────────────────────────────────────────
# Compare before vs after to verify cleaning worked correctly.

snapshot(df_raw,   'BEFORE CLEANING')
snapshot(df_clean, 'AFTER CLEANING')

print(f'\n  First 5 rows after cleaning:')
display(df_clean.head())


── BEFORE CLEANING ──────────────────────────────
  Shape      : (4668, 58)
  Duplicates : 0
  Missing    : 4217 cells (1.56%)
  Dtypes     :
    • Name: str
    • BSE Code: float64
    • NSE Code: str
    • Industry: str
    • Current Price: float64
    • Sales: float64
    • OPM: float64
    • Profit after tax: float64
    • Return on capital employed: float64
    • EPS: float64
    • Change in promoter holding: float64
    • Sales last year: float64
    • Operating profit last year: float64
    • Other income last year: float64
    • EBIDT last year: float64
    • Depreciation last year: float64
    • EBIT last year: float64
    • Interest last year: float64
    • Profit before tax last year: float64
    • Tax last year: float64
    • Profit after tax last year: float64
    • Extraordinary items last year: float64
    • Net Profit last year: float64
    • Dividend last year: float64
    • Material cost last year: float64
    • Employee cost last year: float64
    • OPM last year: f

,name,bse_code,nse_code,industry,current_price,sales,opm,profit_after_tax,return_on_capital_employed,eps,...,extraordinary_items_preceding_year,net_profit_preceding_year,dividend_preceding_year,opm_preceding_year,npm_preceding_year,eps_preceding_year,sales_preceding_12months,net_profit_preceding_12months,market_capitalization,all_metrics_missing
0,20 Microns,533022.0,20MICRONS,Mining / Minerals / Metals,224.55,777.49,13.59,57.38,21.70,15.89,...,0.00,41.96,2.65,12.23,5.98,11.85,738.43,51.80,792.37,False
1,21st Cent. Mgmt.,526921.0,21STCENMGM,Finance & Investments,70.44,34.92,94.27,32.23,73.00,30.70,...,0.00,-15.58,0.00,-115.06,NaN,-14.84,11.94,9.93,73.96,False
2,360 ONE,542772.0,360ONE,Finance & Investments,1009.40,2920.91,58.37,804.18,14.47,22.41,...,0.01,657.89,2457.09,62.77,31.99,18.48,2245.61,717.01,36627.84,False
3,3B Blackbio,532067.0,__NA__,Healthcare,1182.00,74.12,44.50,32.10,20.95,38.13,...,0.00,25.94,1.87,47.12,41.95,34.55,69.24,28.49,1014.49,False
4,3C IT Solutions,544190.0,__NA__,Computers - Software - Medium / Small,44.01,61.93,5.17,1.14,25.63,31.67,...,0.00,1.11,0.00,4.21,1.66,1110.00,NaN,NaN,26.50,False


In [33]:
# ── Cell 8: Cleaning Report ───────────────────────────────────────────────────
# Packages the cleaning results into a report dictionary.
# Pass df_clean + cleaning_report into 03_eda_engine.ipynb

cleaning_report = {
    'filename'         : DATASET_FILENAME,
    'dataset_type'     : dataset_type,
    'rows_before'      : df_raw.shape[0],
    'rows_after'       : df_clean.shape[0],
    'cols_before'      : df_raw.shape[1],
    'cols_after'       : df_clean.shape[1],
    'missing_before'   : int(df_raw.isnull().sum().sum()),
    'missing_after'    : int(df_clean.isnull().sum().sum()),
    'duplicates_removed': int(df_raw.duplicated().sum()),
    'final_columns'    : list(df_clean.columns),
    'final_dtypes'     : df_clean.dtypes.astype(str).to_dict(),
    'final_shape'      : df_clean.shape,
}

print('=' * 60)
print('  CLEANING REPORT')
print('=' * 60)
for k, v in cleaning_report.items():
    if k not in ['final_columns', 'final_dtypes']:
        print(f'  {k:<25}: {v}')
print('\n  Final columns:')
for col in cleaning_report['final_columns']:
    print(f'    • {col}')
print('=' * 60)
print('\n✅ Cleaning complete. Ready for 03_eda_engine.ipynb')

  CLEANING REPORT
  filename                 : Annual_P_L_1_final.csv
  dataset_type             : fundamental
  rows_before              : 4668
  rows_after               : 4668
  cols_before              : 58
  cols_after               : 58
  missing_before           : 4217
  missing_after            : 4217
  duplicates_removed       : 0
  final_shape              : (4668, 58)

  Final columns:
    • name
    • bse_code
    • nse_code
    • industry
    • current_price
    • sales
    • opm
    • profit_after_tax
    • return_on_capital_employed
    • eps
    • change_in_promoter_holding
    • sales_last_year
    • operating_profit_last_year
    • other_income_last_year
    • ebidt_last_year
    • depreciation_last_year
    • ebit_last_year
    • interest_last_year
    • profit_before_tax_last_year
    • tax_last_year
    • profit_after_tax_last_year
    • extraordinary_items_last_year
    • net_profit_last_year
    • dividend_last_year
    • material_cost_last_year
    • employee_co